# Caso guiado Olist · De la teoría a la práctica (Sesiones 1 a 3)
**Maestría en Inteligencia Artificial y Data Science para la Transformación de Negocios · UMSA – Postgrado en Informática**
**Módulo:** Preparación, Recopilación y Análisis Exploratorio de Datos · **Docente:** M.Sc. Marcelo Pinto Macedo

---

### ¿Qué haremos?
Ya construimos la **base conceptual común** del módulo. Ahora la aplicamos, **sesión por sesión**, sobre un conjunto de datos real de comercio electrónico:

| Sección | Viene de | Qué aplicamos |
|---|---|---|
| **S1** | Sesión 1 · Introducción al análisis de datos | Tipos de datos y escalas de medición (Stevens, 1946) |
| **S2** | Sesión 2 · Recopilación de datos | Tamaño de muestra (Cochran, 1977) y muestreo estratificado |
| **S3** | Sesión 3 · Fuentes y extracción | Integración de tablas relacionadas por claves (Codd, 1970) |
| **Cierre** | CRISP-DM, fase 2 | Informe de recolección inicial de datos (Chapman et al., 2000) |

### La pregunta de negocio que guía la práctica
> **¿Qué porcentaje de pedidos llegó tarde y cómo afecta eso a la satisfacción del cliente?**

### Cómo trabajar
- Ejecuta las celdas **en orden** con **Mayús + Enter**.
- En las celdas **✍️ Tu turno**, completa el código donde dice `___`. Hay un ejercicio **básico** y, a veces, un **reto** opcional.
- Si algo falla, escribe tu duda en el chat.

## Paso 0 · El conjunto de datos y cómo lo obtenemos
**Brazilian E-Commerce Public Dataset by Olist.** Olist es un marketplace brasileño. Publicó una muestra real y anonimizada de unos **100 000 pedidos** realizados entre **2016 y 2018**, en varias tablas extraídas **directamente de su sistema transaccional, sin tratamiento previo**.

- **Fuente oficial:** repositorio de Olist en GitHub (`github.com/olist/work-at-olist-data`) y Kaggle (`kaggle.com/datasets/olistbr/brazilian-ecommerce`).
- **Formato:** CSV, uno por tabla.
- **Acceso:** leemos cada archivo directamente desde su **dirección en GitHub** con `pd.read_csv(url)`. No hace falta descargarlo.

Usaremos cuatro tablas: **pedidos**, **clientes**, **ítems** y **reseñas**. La carga puede tardar unos segundos.

> **Plan B (si falla la conexión):** descarga los CSV desde Moodle, súbelos con el ícono de carpeta 📁 (panel izquierdo de Colab) y cambia `BASE` por `""`.

In [ ]:
import math
import pandas as pd

BASE = "https://raw.githubusercontent.com/olist/work-at-olist-data/master/datasets/"

pedidos  = pd.read_csv(BASE + "olist_orders_dataset.csv")
clientes = pd.read_csv(BASE + "olist_customers_dataset.csv")
items    = pd.read_csv(BASE + "olist_order_items_dataset.csv")
resenas  = pd.read_csv(BASE + "olist_order_reviews_dataset.csv")

print("Pedidos: ", pedidos.shape)
print("Clientes:", clientes.shape)
print("Ítems:   ", items.shape)
print("Reseñas: ", resenas.shape)

**Resultado esperado:** pedidos (99441, 8) · clientes (99441, 5) · ítems (112650, 7) · reseñas (99224, 7).

Observa que hay **más ítems que pedidos**: un pedido puede tener varios productos.

---
# S1 · De la Sesión 1: tipos de datos y escalas de medición
**Recordemos:** Stevens (1946) propuso cuatro escalas — **nominal, ordinal, intervalo y razón** — y la escala determina qué cálculos son válidos.

### S1.1 · Primera mirada a los pedidos
`head()` muestra las primeras filas e `info()` el tipo de dato de cada columna.

In [ ]:
pedidos.head(3)

In [ ]:
pedidos.info()

**Qué observar:** las cinco columnas de fechas aparecen como **texto** (`object` o `str`, según la versión de pandas), no como fechas. Además, algunas fechas tienen **valores faltantes**: el conteo de «non-null» es menor que 99 441.

Mientras una fecha sea texto, no podemos restar fechas ni calcular retrasos. **Convertimos** las columnas con `pd.to_datetime`:

In [ ]:
columnas_fecha = ["order_purchase_timestamp", "order_approved_at", "order_delivered_carrier_date",
                  "order_delivered_customer_date", "order_estimated_delivery_date"]
for col in columnas_fecha:
    pedidos[col] = pd.to_datetime(pedidos[col])

print(pedidos[columnas_fecha].dtypes)
print("Primer pedido:", pedidos["order_purchase_timestamp"].min())
print("Último pedido:", pedidos["order_purchase_timestamp"].max())

**Resultado esperado:** las columnas pasan a `datetime64`; el primer pedido es del **4 de septiembre de 2016** y el último del **17 de octubre de 2018**.

### S1.2 · Un número que no es un número
En la tabla de clientes, el **código postal** se leyó como número entero. Mira las primeras filas:

In [ ]:
clientes[["customer_zip_code_prefix", "customer_city", "customer_state"]].head(3)

**Qué observar:** el tercer cliente aparece con código **1151**, pero el código real es **01151**: al leerlo como número, **se perdió el cero inicial**.

El código postal es una variable **nominal**: identifica, no mide. Como vimos en la Sesión 1 con el número de carnet, **sumarlo o promediarlo no tiene sentido**, y debe guardarse como **texto**.

In [ ]:
clientes["customer_zip_code_prefix"] = clientes["customer_zip_code_prefix"].astype(str).str.zfill(5)
clientes[["customer_zip_code_prefix"]].head(3)

### S1.3 · Escala de cada variable
| Variable | Tabla | Escala | Cálculos válidos |
|---|---|---|---|
| `order_status` (estado del pedido) | pedidos | Nominal | Frecuencias, moda |
| `customer_state` (estado de Brasil) | clientes | Nominal | Frecuencias, moda |
| `customer_zip_code_prefix` | clientes | Nominal | Frecuencias |
| `review_score` (1 a 5 estrellas) | reseñas | **Ordinal** | Mediana, percentiles |
| `order_purchase_timestamp` (fecha) | pedidos | Intervalo | Diferencias entre fechas |
| `price`, `freight_value` (precio, flete) | ítems | **Razón** | Todos, incluidas razones |

Una variable **nominal** se resume con frecuencias:

In [ ]:
pedidos["order_status"].value_counts()

**Resultado esperado:** `delivered` 96 478 · `shipped` 1 107 · `canceled` 625 · … El **97 %** de los pedidos fue entregado.

Una variable de **razón** se resume con estadísticos:

In [ ]:
items[["price", "freight_value"]].describe().round(2)

**Resultado esperado:** precio medio **120,65** y mediana (50 %) **74,99**. La media es mucho mayor que la mediana porque hay **precios muy altos** (máximo 6 735): una pista de valores atípicos, que trabajaremos en el análisis exploratorio.

> ### ✍️ Tu turno · S1 · básico
> Muestra cuántas reseñas hay de cada puntaje (`review_score`), ordenadas de 1 a 5.
>
> **Pista:** usa `value_counts()` y luego `sort_index()`.
> **Resultado esperado:** 1 → 11 424 · 2 → 3 151 · 3 → 8 179 · 4 → 19 142 · 5 → 57 328

In [ ]:
resenas["review_score"].___().___()

> ### ✍️ Tu turno · S1 · reto (opcional)
> Calcula la **media** y la **mediana** de `review_score`. ¿Cuál es más adecuada según la escala de Stevens y por qué?
>
> **Pista:** `mean()` y `median()`. Recuerda que `review_score` es ordinal.
> **Resultado esperado:** media 4,09 · mediana 5. Para una escala ordinal, lo riguroso es la **mediana** (y la distribución de frecuencias).

In [ ]:
print("Media:  ", resenas["review_score"].___())
print("Mediana:", resenas["review_score"].___())

---
# S2 · De la Sesión 2: tamaño de muestra y muestreo
**Recordemos:** la fórmula de **Cochran (1977)** con corrección por población finita, y la ventaja del **muestreo estratificado** para representar bien a los grupos pequeños.

### S2.1 · ¿Cuántas reseñas habría que revisar?
Supongamos que Olist quiere leer a mano una muestra de reseñas para clasificar los motivos de queja. La población es de **99 224 reseñas**.

In [ ]:
def tamano_muestra(N, e=0.05, z=1.96, p=0.5):
    n0 = z**2 * p * (1 - p) / e**2               # muestra para población muy grande
    return math.ceil(n0 / (1 + (n0 - 1) / N))    # corrección por población finita

n = tamano_muestra(len(resenas))
n

**Resultado esperado:** **383** reseñas (95 % de confianza y 5 % de error).

### S2.2 · Aleatoria simple frente a estratificada
Los puntajes **no** están equilibrados: las reseñas de **2 estrellas** son solo el **3,2 %**. Comparamos cómo las representa cada muestra:

In [ ]:
aleatoria = resenas.sample(n=n, random_state=42)
estratificada = resenas.groupby("review_score", group_keys=False).sample(frac=n / len(resenas), random_state=42)

comparacion = pd.DataFrame({
    "Población": resenas["review_score"].value_counts(normalize=True),
    "Aleatoria": aleatoria["review_score"].value_counts(normalize=True),
    "Estratificada": estratificada["review_score"].value_counts(normalize=True),
}).sort_index().round(3)
comparacion

**Resultado esperado** para las reseñas de 2 estrellas: población **0,032** · aleatoria **0,023** · estratificada **0,031**.

**Conclusión:** la estratificada reproduce la población; la aleatoria subrepresentó a las reseñas de 2 estrellas, que justamente son las quejas que más interesa leer.

> ### ✍️ Tu turno · S2 · básico
> ¿Cuántas reseñas habría que revisar con un margen de error del **3 %**?
>
> **Pista:** usa la función `tamano_muestra` con el argumento `e=0.03`.
> **Resultado esperado:** 1 056 reseñas. Reducir el error del 5 % al 3 % casi triplica la muestra.

In [ ]:
tamano_muestra(len(resenas), e=___)

---
# S3 · De la Sesión 3: fuentes e integración de tablas
**Recordemos:** en una base de datos relacional (**Codd, 1970**) cada tabla guarda una entidad y las tablas se **relacionan por claves**.

### S3.1 · El modelo de datos de Olist
```
clientes ──(customer_id)── pedidos ──(order_id)── ítems
                               │
                          (order_id)
                               │
                            reseñas
```
- `customer_id` une **pedidos** con **clientes**.
- `order_id` une **pedidos** con **ítems** y con **reseñas**.

### S3.2 · ¿De qué estados de Brasil son los pedidos?
El estado del cliente está en otra tabla. Las **unimos** con `merge`, como un `JOIN` de SQL:

In [ ]:
pedidos_clientes = pedidos.merge(clientes, on="customer_id", how="left")
print(pedidos_clientes.shape)
pedidos_clientes["customer_state"].value_counts().head(5)

**Resultado esperado:** (99441, 12) y, en primer lugar, **SP** (São Paulo) con 41 746 pedidos, luego RJ con 12 852 y MG con 11 635.

### S3.3 · La pregunta de negocio: ¿cuántos pedidos llegaron tarde?
Un pedido llegó tarde si la **fecha real de entrega** es posterior a la **fecha estimada**. Usamos solo los pedidos entregados que tienen fecha de entrega registrada.

In [ ]:
entregados = pedidos[pedidos["order_status"] == "delivered"].dropna(subset=["order_delivered_customer_date"]).copy()
entregados["tarde"] = entregados["order_delivered_customer_date"] > entregados["order_estimated_delivery_date"]

print("Pedidos entregados analizados:", len(entregados))
print("Llegaron tarde:", entregados["tarde"].sum())
print("Porcentaje tarde:", round(entregados["tarde"].mean() * 100, 1), "%")

**Resultado esperado:** 96 470 pedidos analizados · 7 826 tarde · **8,1 %**.

> ### ✍️ Tu turno · S3 · básico
> Une **pedidos** con **reseñas** usando la clave común. ¿Cuántas filas tiene el resultado? ¿Por qué son **más** que los 99 441 pedidos?
>
> **Pista:** la clave es `order_id`; usa `how="left"`.
> **Resultado esperado:** (99992, 14). Hay más filas porque algunos pedidos tienen **más de una reseña**: al unir, se repiten. Es un problema de calidad que retomaremos en la limpieza.

In [ ]:
pedidos_resenas = pedidos.merge(resenas, on="___", how="left")
pedidos_resenas.shape

> ### ✍️ Tu turno · S3 · reto (opcional)
> ¿Cuál es el puntaje promedio de las reseñas de los pedidos que llegaron **tarde** y de los que llegaron **a tiempo**?
>
> **Pista:** une `entregados` con `resenas[["order_id", "review_score"]]` y agrupa por la columna `tarde`.
> **Resultado esperado:** a tiempo (False) **4,29** · tarde (True) **2,57**. ¡Llegar tarde baja la satisfacción en casi dos estrellas!

In [ ]:
x = entregados.merge(resenas[["order_id", "review_score"]], on="order_id")
x.groupby("___")["review_score"].mean().round(2)

---
# Cierre · Informe de recolección inicial (CRISP-DM, fase 2)
La guía CRISP-DM (Chapman et al., 2000) pide documentar la tarea **«Recolectar los datos iniciales»**. Este es el informe del caso Olist:

| Elemento | Caso Olist |
|---|---|
| **Fuente** | Olist, marketplace brasileño · repositorio oficial en GitHub y Kaggle |
| **Tipo de fuente** | Externa · datos secundarios · extraídos de un sistema transaccional |
| **Formato y acceso** | CSV · lectura directa por URL |
| **Tablas usadas** | pedidos (99 441) · clientes (99 441) · ítems (112 650) · reseñas (99 224) |
| **Periodo** | 4 de septiembre de 2016 a 17 de octubre de 2018 |
| **Problemas encontrados** | Fechas leídas como texto · faltantes en fechas de entrega · códigos postales sin cero inicial · pedidos con más de una reseña · comentarios vacíos en muchas reseñas |
| **Primer hallazgo** | El 8,1 % de los pedidos entregados llegó tarde, con 2,57 estrellas promedio frente a 4,29 de los que llegaron a tiempo |

### Para tu proyecto
Repite estas mismas secciones con **tu conjunto de datos** en la **plantilla espejo** (`Mi_proyecto_De_la_teoria_a_la_practica.ipynb`).

---
**Referencias:** Chapman, P. et al. (2000). *CRISP-DM 1.0*. SPSS. · Cochran, W. G. (1977). *Sampling techniques* (3.ª ed.). Wiley. · Codd, E. F. (1970). A relational model of data for large shared data banks. *Communications of the ACM*, 13(6), 377–387. · Olist (2018). *Brazilian E-Commerce Public Dataset by Olist*. Kaggle / GitHub. · Stevens, S. S. (1946). On the theory of scales of measurement. *Science*, 103(2684), 677–680.